In [1]:
import os
import re
import sys
import numpy as np
import pandas as pd

sys.path.append(os.path.abspath("../../")) ; from EPF import variables
sys.path.insert(0, os.path.abspath("../../../Generic-Parallel-Compute-Helper/")) ; from memory_compute import *

# Detect free RAM now, cap this kernel so it can never exhaust the machine, and
# make every parquet read/write below stream in bounded batches instead of
# loading the whole frame at once.
install_memory_guard()

REGION        = variables.TARGET_REGION
ALL_REGIONS   = ["nsw", "qld", "vic", "sa"]
OTHER_REGIONS = [r for r in ALL_REGIONS if r != REGION]

# PDPASA REGIONSOLUTION forecast. pdpasa_{value}_{region}_h{k} is the forecast
# for the k-th 30-min period after each timestamp, from the run available then
# (leakage-free ex-ante). All values share one consolidated parquet
# (7_1_pdpasa_regionsolution). Values: demand10/50/90, reservereq,
# surplusreserve, maxsurplusreserve, maxsparecapacity, aggregatepasaavailability.

SRC_PATH = "../1_Dataset/Processed_data/7_1_pdpasa_regionsolution.parquet"
OUT_PATH = "../2_Features_build/Feature_data/7_pdpasa_region_solution.parquet"


def _pdpasa_cols(df: pd.DataFrame, value: str, region: str) -> list:
    """Forecast columns for a (value, region), ordered by horizon."""
    prefix = f"pdpasa_{value}_{region}_h"
    found = [
        (int(re.search(r"_h(\d+)$", c).group(1)), c)
        for c in df.columns
        if c.startswith(prefix)
    ]
    return [c for _, c in sorted(found)]


[memory_guard] hard cap 13.3G virtual on this kernel (total RAM 14.8G, 6.5G free now). Runaway allocations fail cleanly; bounded streaming keeps normal work well under this.


In [2]:
# The final cell streams the source in bounded row-batches straight to disk
# (dropping duplicate columns per batch via dedup_columns=True). Here we only
# pull a tiny sample so the feature functions below can be previewed without
# ever holding the full frame in memory.
sample = peek_parquet(SRC_PATH, 10)
sample = sample.loc[:, ~sample.columns.duplicated()]
sample.iloc[:, :6]


,pdpasa_demand10_nsw_h1,pdpasa_demand10_nsw_h2,pdpasa_demand10_nsw_h3,pdpasa_demand10_nsw_h4,pdpasa_demand10_nsw_h5,pdpasa_demand10_nsw_h6
Date,,,,,,
2018-01-01 00:00:00,7082.0,6881.0,6606.0,6389.0,6242.0,6151.0
2018-01-01 00:05:00,7082.0,6881.0,6606.0,6389.0,6242.0,6151.0
2018-01-01 00:10:00,7082.0,6881.0,6606.0,6389.0,6242.0,6151.0
2018-01-01 00:15:00,7082.0,6881.0,6606.0,6389.0,6242.0,6151.0
2018-01-01 00:20:00,7082.0,6881.0,6606.0,6389.0,6242.0,6151.0
2018-01-01 00:25:00,7082.0,6881.0,6606.0,6389.0,6242.0,6151.0
2018-01-01 00:30:00,6868.0,6598.0,6384.0,6244.0,6157.0,6112.0
2018-01-01 00:35:00,6868.0,6598.0,6384.0,6244.0,6157.0,6112.0
2018-01-01 00:40:00,6868.0,6598.0,6384.0,6244.0,6157.0,6112.0


In [3]:
def _add_pdpasa_reserve_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Forecast reserve adequacy for the target region from PDPASA. The gap between
    aggregate PASA availability and forecast (P50) demand is AEMO's own
    look-ahead scarcity signal; a thin forecast reserve at any horizon in the
    next 24h is a leading spike indicator. Leakage-free forecasts.
    Returns only the new columns to avoid copying the full base frame.
    """
    R = REGION
    new_cols = {}
    avail = _pdpasa_cols(df, "aggregatepasaavailability", R)
    dem50 = _pdpasa_cols(df, "demand50", R)
    if avail and dem50:
        n = min(len(avail), len(dem50))
        A = df[avail[:n]].to_numpy()
        D = df[dem50[:n]].to_numpy()
        margin = pd.DataFrame((A - D) / (D + 1), index=df.index)
        h24 = min(48, n)
        new_cols[f"pdpasa_reserve_margin_{R}_fh1"]      = margin.iloc[:, 0].astype(np.float32)
        new_cols[f"pdpasa_reserve_margin_{R}_min_24h"]  = margin.iloc[:, :h24].min(axis=1).astype(np.float32)
        new_cols[f"pdpasa_reserve_margin_{R}_min_full"] = margin.min(axis=1).astype(np.float32)
        new_cols[f"pdpasa_reserve_margin_{R}_mean_24h"] = margin.iloc[:, :h24].mean(axis=1).astype(np.float32)
    return pd.DataFrame(new_cols, index=df.index)


_add_pdpasa_reserve_features(sample)[:10]


,pdpasa_reserve_margin_nsw_fh1,pdpasa_reserve_margin_nsw_min_24h,pdpasa_reserve_margin_nsw_min_full,pdpasa_reserve_margin_nsw_mean_24h
Date,,,,
2018-01-01 00:00:00,0.873339,0.477059,0.477059,0.812113
2018-01-01 00:05:00,0.873339,0.477059,0.477059,0.812113
2018-01-01 00:10:00,0.873339,0.477059,0.477059,0.812113
2018-01-01 00:15:00,0.873339,0.477059,0.477059,0.812113
2018-01-01 00:20:00,0.873339,0.477059,0.477059,0.812113
2018-01-01 00:25:00,0.873339,0.477059,0.477059,0.812113
2018-01-01 00:30:00,0.931646,0.472667,0.472667,0.806812
2018-01-01 00:35:00,0.931646,0.472667,0.472667,0.806812
2018-01-01 00:40:00,0.931646,0.472667,0.472667,0.806812


In [4]:
def _add_pdpasa_headroom_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Forecast headroom and demand uncertainty for the target region: minimum
    surplus reserve and spare capacity over the next 24h, and the P90-P50
    demand band (forecast demand uncertainty, which widens ahead of spikes).
    Leakage-free forecasts.
    Returns only the new columns to avoid copying the full base frame.
    """
    R = REGION
    new_cols = {}
    for value, label in [("surplusreserve", "surplus"), ("maxsparecapacity", "sparecap")]:
        cols = _pdpasa_cols(df, value, R)
        if cols:
            sub = df[cols]
            h24 = min(48, sub.shape[1])
            new_cols[f"pdpasa_{label}_{R}_min_24h"] = sub.iloc[:, :h24].min(axis=1).astype(np.float32)

    d90 = _pdpasa_cols(df, "demand90", R)
    d50 = _pdpasa_cols(df, "demand50", R)
    if d90 and d50:
        n = min(len(d90), len(d50))
        band = pd.DataFrame(df[d90[:n]].to_numpy() - df[d50[:n]].to_numpy(), index=df.index)
        h24 = min(48, n)
        new_cols[f"pdpasa_demand_band_{R}_max_24h"] = band.iloc[:, :h24].max(axis=1).astype(np.float32)

    return pd.DataFrame(new_cols, index=df.index)


_add_pdpasa_headroom_features(sample)[:10]


,pdpasa_surplus_nsw_min_24h,pdpasa_sparecap_nsw_min_24h,pdpasa_demand_band_nsw_max_24h
Date,,,
2018-01-01 00:00:00,4351.709961,3803.560059,-160.0
2018-01-01 00:05:00,4351.709961,3803.560059,-160.0
2018-01-01 00:10:00,4351.709961,3803.560059,-160.0
2018-01-01 00:15:00,4351.709961,3803.560059,-160.0
2018-01-01 00:20:00,4351.709961,3803.560059,-160.0
2018-01-01 00:25:00,4351.709961,3803.560059,-160.0
2018-01-01 00:30:00,4329.790039,3760.629883,-155.0
2018-01-01 00:35:00,4329.790039,3760.629883,-155.0
2018-01-01 00:40:00,4329.790039,3760.629883,-155.0


In [5]:
def _add_pdpasa_neighbour_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Neighbouring regions' tightest forecast reserve margin over the next 24h. A
    scarcity event next door pulls exports through the interconnectors and lifts
    the target region's price too. Leakage-free forecasts.
    Returns only the new columns to avoid copying the full base frame.
    """
    new_cols = {}
    for r in OTHER_REGIONS:
        avail = _pdpasa_cols(df, "aggregatepasaavailability", r)
        dem50 = _pdpasa_cols(df, "demand50", r)
        if avail and dem50:
            n = min(len(avail), len(dem50))
            margin = pd.DataFrame(
                (df[avail[:n]].to_numpy() - df[dem50[:n]].to_numpy()) / (df[dem50[:n]].to_numpy() + 1),
                index=df.index,
            )
            h24 = min(48, n)
            new_cols[f"pdpasa_reserve_margin_{r}_min_24h"] = margin.iloc[:, :h24].min(axis=1).astype(np.float32)
    return pd.DataFrame(new_cols, index=df.index)


_add_pdpasa_neighbour_features(sample)[:10]


,pdpasa_reserve_margin_qld_min_24h,pdpasa_reserve_margin_vic_min_24h,pdpasa_reserve_margin_sa_min_24h
Date,,,
2018-01-01 00:00:00,0.456595,1.047545,0.997794
2018-01-01 00:05:00,0.456595,1.047545,0.997794
2018-01-01 00:10:00,0.456595,1.047545,0.997794
2018-01-01 00:15:00,0.456595,1.047545,0.997794
2018-01-01 00:20:00,0.456595,1.047545,0.997794
2018-01-01 00:25:00,0.456595,1.047545,0.997794
2018-01-01 00:30:00,0.458473,1.049832,0.997794
2018-01-01 00:35:00,0.458473,1.049832,0.997794
2018-01-01 00:40:00,0.458473,1.049832,0.997794


In [6]:
def add_pdpasa_features(df: pd.DataFrame) -> pd.DataFrame:
    # All groups are row-wise over the horizon columns, so they compute
    # identically on a row-batch as on the full frame.
    return pd.concat(
        [
            _add_pdpasa_reserve_features(df),
            _add_pdpasa_headroom_features(df),
            _add_pdpasa_neighbour_features(df),
        ],
        axis=1,
    )


# Retain core columns (keep_source=True): the PDPASA forecast curves are ex-ante
# forecasts from the run available at t -> leakage-free, used unshifted.
# dedup_columns=True reproduces the original duplicate-column guard per batch.
# Streams source + new features to disk in bounded batches; peak RAM is one
# batch, so this cannot exhaust memory.
stream_transform_parquet(
    SRC_PATH, OUT_PATH, transform=add_pdpasa_features, keep_source=True, dedup_columns=True
)

import pyarrow.parquet as pq
meta = pq.ParquetFile(OUT_PATH).metadata
print("Total features:", meta.num_columns)
(meta.num_rows, meta.num_columns)


[stream] 7_1_pdpasa_regionsolution.parquet: 893,665 rows x 3121 cols, largest row group 3.21G -> column-block strategy (bounded regardless of free RAM)


Merge rows:  46%|████▌     | 76/167 [01:12<01:24,  1.08batch/s]/tmp/ipykernel_148054/1696617039.py:15: RuntimeWarning: divide by zero encountered in divide
  (df[avail[:n]].to_numpy() - df[dem50[:n]].to_numpy()) / (df[dem50[:n]].to_numpy() + 1),
Merge rows:  47%|████▋     | 78/167 [01:14<01:24,  1.05batch/s]/tmp/ipykernel_148054/1696617039.py:15: RuntimeWarning: divide by zero encountered in divide
  (df[avail[:n]].to_numpy() - df[dem50[:n]].to_numpy()) / (df[dem50[:n]].to_numpy() + 1),
Merge rows:  56%|█████▋    | 94/167 [01:30<01:11,  1.02batch/s]/tmp/ipykernel_148054/1696617039.py:15: RuntimeWarning: divide by zero encountered in divide
  (df[avail[:n]].to_numpy() - df[dem50[:n]].to_numpy()) / (df[dem50[:n]].to_numpy() + 1),
Merge rows:  67%|██████▋   | 112/167 [01:47<00:51,  1.08batch/s]/tmp/ipykernel_148054/1696617039.py:15: RuntimeWarning: divide by zero encountered in divide
  (df[avail[:n]].to_numpy() - df[dem50[:n]].to_numpy()) / (df[dem50[:n]].to_numpy() + 1),
Merge rows:  68

[stream] wrote 893,665 rows -> ../2_Features_build/Feature_data/7_pdpasa_region_solution.parquet
Total features: 3131


(893665, 3131)

In [ ]:
# Free this kernel's memory so the next notebook has RAM to work with
# (clears data variables + returns freed heap to the OS).
release_memory()
